# Momentum Strategy Walkthrough

This notebook runs the full project in inspectable steps:

1. Refresh raw data
2. Inspect the universe and price panel
3. Build features
4. Score the cross-section
5. Run the backtest
6. Write reports and inspect saved outputs

It is meant to be run from inside this repository. If your WRDS credentials are configured, the notebook will use the WRDS path automatically.

In [ ]:
from __future__ import annotations

import argparse
import json
import sys
from datetime import date
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd()
if project_root.name.lower() == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src import main as project_main
from src.config import (
    BACKTEST_SUMMARY_PATH,
    FEATURE_PANEL_PATH,
    PIPELINE_SUMMARY_PATH,
    PRICE_PANEL_PATH,
    PROCESSED_UNIVERSE_SOURCE_PATH,
    RUN_MANIFEST_PATH,
    SCORE_PANEL_PATH,
    SCREENER_SUMMARY_PATH,
    UNIVERSE_MEMBERSHIP_PATH,
    UNIVERSE_SOURCE_PATH,
    ensure_output_dirs,
)

ensure_output_dirs()
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)


def show_json(payload: dict[str, object]) -> None:
    print(json.dumps(payload, indent=2))


def load_frame(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported file type: {path}")


print(f"Project root: {project_root}")

In [ ]:
START = "2018-01-01"
END = str(date.today())
LIMIT = 300
UNIVERSE_MODE = "top_liquid"

print({
    "start": START,
    "end": END,
    "limit": LIMIT,
    "universe_mode": UNIVERSE_MODE,
})

## 1. Refresh Raw Data

In [ ]:
refresh_summary = project_main.refresh_data(
    start=START,
    end=END,
    limit=LIMIT,
    universe_mode=UNIVERSE_MODE,
)
show_json(refresh_summary)

In [ ]:
universe_source = load_frame(UNIVERSE_SOURCE_PATH)
price_panel = load_frame(PRICE_PANEL_PATH)

print("Universe source shape:", universe_source.shape)
display(universe_source.head(10))

print("Price panel shape:", price_panel.shape)
print("Date range:", pd.to_datetime(price_panel["date"]).min().date(), "->", pd.to_datetime(price_panel["date"]).max().date())
print("Ticker count:", price_panel["ticker"].nunique())
display(price_panel.head(10))

## 2. Build Features

In [ ]:
feature_summary = project_main.build_features(limit=LIMIT)
show_json(feature_summary)

In [ ]:
processed_universe = load_frame(PROCESSED_UNIVERSE_SOURCE_PATH)
universe_membership = load_frame(UNIVERSE_MEMBERSHIP_PATH)
feature_panel = load_frame(FEATURE_PANEL_PATH)

latest_feature_date = pd.to_datetime(feature_panel["date"]).max()
latest_membership = universe_membership[pd.to_datetime(universe_membership["date"]) == latest_feature_date]

print("Processed universe snapshot shape:", processed_universe.shape)
print("Universe membership shape:", universe_membership.shape)
print("Feature panel shape:", feature_panel.shape)
print("Latest feature date:", latest_feature_date.date())

display(latest_membership[["ticker", "sector", "universe_rank", "median_dollar_volume_60"]].head(20))
display(feature_panel[["date", "ticker", "mom_12_1", "mom_6_1", "risk_adj_mom", "prox_52w_high", "avg_dollar_volume_60"]].tail(20))

## 3. Score The Cross-Section

In [ ]:
score_summary = project_main.score_universe()
show_json(score_summary)

In [ ]:
score_panel = load_frame(SCORE_PANEL_PATH)
latest_score_date = pd.to_datetime(score_panel["date"]).max()
latest_scores = score_panel[pd.to_datetime(score_panel["date"]) == latest_score_date].sort_values("core_rank")

print("Score panel shape:", score_panel.shape)
print("Latest score date:", latest_score_date.date())
display(latest_scores[["ticker", "sector", "core_rank", "core_rank_pct", "core_score", "experimental_score", "mom_6_1", "mom_12_1"]].head(30))

## 4. Run The Backtest

In [ ]:
backtest_summary = project_main.backtest_phase()
show_json(backtest_summary)

## 5. Write Reports And Inspect Saved Outputs

In [ ]:
report_summary = project_main.report_phase()
manifest_args = argparse.Namespace(start=START, end=END, limit=LIMIT, universe_mode=UNIVERSE_MODE)
project_main._write_run_manifest(
    manifest_args,
    "notebook",
    {
        "refresh": refresh_summary,
        "features": feature_summary,
        "scores": score_summary,
        "backtest": backtest_summary,
        "report": report_summary,
    },
)
show_json(report_summary)

In [ ]:
with open(BACKTEST_SUMMARY_PATH, "r", encoding="utf-8") as handle:
    backtest_metrics = json.load(handle)
with open(SCREENER_SUMMARY_PATH, "r", encoding="utf-8") as handle:
    screener_metrics = json.load(handle)
with open(PIPELINE_SUMMARY_PATH, "r", encoding="utf-8") as handle:
    pipeline_metrics = json.load(handle)
with open(RUN_MANIFEST_PATH, "r", encoding="utf-8") as handle:
    run_manifest = json.load(handle)

print("Backtest metrics")
show_json(backtest_metrics)

print("Screener metrics")
show_json(screener_metrics)

print("Pipeline summary")
show_json(pipeline_metrics)

print("Run manifest keys:", list(run_manifest.keys()))
display(pd.DataFrame(run_manifest["inputs"]).T)